In [5]:
#Import models and libraries
from DT.DecisionTree import DecisionTree
from sklearn.tree import DecisionTreeClassifier
from GNB.gaussian_naive_bayes import GNB
from LogisticRegresssion.LogisticRegression import LogReg
from tensorflow import keras
from SVM.linear_svm import LinearSVMScartch
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report , f1_score
from sklearn.model_selection import train_test_split



ModuleNotFoundError: No module named 'sklearn'

In [19]:
#Download Data
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()


In [20]:
y_train_bin = (y_train == 6).astype(int)
y_test_bin  = (y_test == 6).astype(int)


In [21]:
# Flatten
# X_train = X_train.reshape(X_train.shape[0], -1)
# X_test = X_test.reshape(X_test.shape[0], -1)

In [22]:
# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

In [23]:
def compute_pixel_variance(X):
    mean = np.mean(X, axis=0)
    return np.mean((X - mean) ** 2, axis=0)

def variance_threshold(X, threshold=1e-4):
    variances = compute_pixel_variance(X)
    mask = variances > threshold
    return X[:, mask], mask

X_train_clean, mask = variance_threshold(X_train)
X_test_clean = X_test[:, mask]

print("After variance filtering:", X_train_clean.shape)


After variance filtering: (60000, 623)


In [24]:
import numpy as np
from skimage.feature import hog

def extract_hog_features(X):
    features = []

    for img in X:
        img_2d = img.reshape(28, 28)

        hog_features = hog(
            img_2d,
            orientations=9,
            pixels_per_cell=(4, 4),
            cells_per_block=(2, 2),
            block_norm='L2-Hys',
            feature_vector=True
        )

        features.append(hog_features)

    return np.array(features)

In [25]:
X_train_HOG = extract_hog_features(X_train)
X_test_HOG = extract_hog_features(X_test)

In [26]:
pca = PCA(n_components=50)  

X_train_hog_pca = pca.fit_transform(X_train_HOG)
X_test_hog_pca = pca.transform(X_test_HOG)

In [27]:
print(X_test_hog_pca.shape)

(10000, 50)


In [28]:
#Decision Tree results
dt = DecisionTree(maxDepth = 12,
minSamplesSplit = 10,
minSampleLeafs = 5,
criterion = "entropy",
maxFeatures = "sqrt",
classWeights = {0: 1, 1: 5}
)


dt.fit(X_train_hog_pca , y_train_bin)
predictions = dt.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin, predictions,
    target_names=["Not 6", "Is 6"]
))



              precision    recall  f1-score   support

       Not 6       0.99      0.98      0.99      9042
        Is 6       0.83      0.93      0.88       958

    accuracy                           0.98     10000
   macro avg       0.91      0.95      0.93     10000
weighted avg       0.98      0.98      0.98     10000



In [29]:
from sklearn.model_selection import KFold
def run_kfold(X, y, train_fn, predict_fn, k=5):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    scores = []

    for i, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train_fold = X[train_idx]
        X_val_fold = X[val_idx]

        y_train_fold = y[train_idx]
        y_val_fold = y[val_idx]

        model = train_fn(X_train_fold, y_train_fold)
        preds = predict_fn(model, X_val_fold)

        score = f1_score(y_val_fold, preds, pos_label=1)
        scores.append(score)

        print(f"Fold {i+1} → F1: {score:.4f}")

    avg = np.mean(scores)
    print("\nK-Fold Avg F1:", avg)

    return avg, scores

In [30]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_hog_pca, y_train_bin,
    test_size=0.2,
    stratify=y_train_bin,
    random_state=42
)

# Train
gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

# Tune weights
best_weight = None
best_score = -1

for w in [1, 1.5, 2, 3, 4, 5, 7, 10]:
    class_weights = {0: 1.0, 1: w}
    preds = gnb.predict(X_val, class_weights=class_weights)
    score = f1_score(y_val, preds, pos_label=1)

    print(f"Weight {w} → F1: {score:.4f}")

    if score > best_score:
        best_score = score
        best_weight = w

print("\n🔥 Best weight:", best_weight)

def train_gnb(X, y):
    gnb = GNB()
    gnb.gaussian_naive_train(X, y)
    return gnb


def predict_gnb(model, X):
    return model.predict(
        X,
        class_weights={0: 1.0, 1: best_weight}
    )


# ======================
# STEP 2: K-FOLD
# ======================
print("\n=== K-FOLD VALIDATION ===")
run_kfold(X_train, y_train, train_gnb, predict_gnb, k=5)


# ======================
# STEP 3: FINAL TEST
# ======================
print("\n=== FINAL TEST RESULTS ===")

gnb = GNB()
gnb.gaussian_naive_train(X_train, y_train)

predictions = gnb.predict(
    X_test_hog_pca,   # ✅ CORRECT
    class_weights={0: 1.0, 1: best_weight}
)

print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

Weight 1 → F1: 0.9309
Weight 1.5 → F1: 0.9392
Weight 2 → F1: 0.9405
Weight 3 → F1: 0.9485
Weight 4 → F1: 0.9504
Weight 5 → F1: 0.9510
Weight 7 → F1: 0.9525
Weight 10 → F1: 0.9536

🔥 Best weight: 10

=== K-FOLD VALIDATION ===
Fold 1 → F1: 0.9472
Fold 2 → F1: 0.9548
Fold 3 → F1: 0.9574
Fold 4 → F1: 0.9525
Fold 5 → F1: 0.9529

K-Fold Avg F1: 0.9529874832956813

=== FINAL TEST RESULTS ===
              precision    recall  f1-score   support

       Not 6       0.99      1.00      1.00      9042
        Is 6       0.98      0.94      0.96       958

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000



In [31]:
lg = LogReg(max_iterations=1000 , learning_rate=0.1 , threshold=0.5)
lg.fit(X_train_hog_pca , y_train_bin)
predictions = lg.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

              precision    recall  f1-score   support

       Not 6       0.99      1.00      1.00      9042
        Is 6       0.99      0.94      0.96       958

    accuracy                           0.99     10000
   macro avg       0.99      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000



In [32]:
y_train_bin = np.where(y_train == 6, 1, -1)
y_test_bin  = np.where(y_test == 6, 1, -1)

In [33]:
svm = LinearSVMScartch(
    C=1.0,
    learning_rate=0.0001,
    n_epochs=100,
    batch_size=128,
    use_class_weights=True,
    random_state=42
)
svm.fit(X_train_hog_pca , y_train_bin)
predictions = svm.predict(X_test_hog_pca)
print(classification_report(
    y_test_bin,
    predictions,
    target_names=["Not 6", "Is 6"]
))

/Users/amirtamer/CAIE/Sem 6/ML/Project/SVM/linear_svm.py:32: RuntimeWarning: divide by zero encountered in scalar divide
  weight_pos = n_samples / (2.0 * n_pos)


IndexError: index 51012 is out of bounds for axis 0 with size 48000

In [ ]:
import numpy as np
from collections import Counter

class CustomKNN:
    def __init__(self, k=5, weights='uniform'):
        """
        k: Number of nearest neighbors to use.
        weights: 'uniform' (standard majority vote) or 'distance' (closer neighbors have a stronger vote).
        """
        self.k = k
        self.weights = weights

    def fit(self, X, y):
        """
        KNN is a lazy learner; training simply memorizes the dataset.
        """
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def predict(self, X_test):
        """
        Predicts labels for an array of test vectors.
        """
        X_test = np.array(X_test)
        # Process each test image one by one
        return np.array([self._predict_single(x) for x in X_test])

    def _predict_single(self, x_test):
        """
        Core logic: calculates distance to all training points and finds the top K.
        """
        # 1. Calculate Euclidean distance against all training vectors using fast NumPy broadcasting
        distances = np.sqrt(np.sum((self.X_train - x_test) ** 2, axis=1))

        # 2. Get the indices of the K smallest distances
        k_indices = np.argsort(distances)[:self.k]
        
        # 3. Extract the actual labels of those K nearest neighbors
        k_nearest_labels = self.y_train[k_indices]
        
        # 4. Voting Logic
        if self.weights == 'distance':
            # Extract the actual distances of the K neighbors
            k_distances = distances[k_indices]
            
            # Weight = 1 / distance (adding 1e-5 to prevent division by zero if distance is exactly 0)
            vote_weights = 1.0 / (k_distances + 1e-5) 
            
            # Dictionary to tally the weighted votes for each class
            class_votes = {}
            for label, weight in zip(k_nearest_labels, vote_weights):
                class_votes[label] = class_votes.get(label, 0) + weight
                
            # Return the class label that accumulated the highest total weight
            return max(class_votes, key=class_votes.get)
            
        else:
            # Standard uniform voting (Majority Vote)
            most_common = Counter(k_nearest_labels).most_common(1)
            return most_common[0][0]

In [ ]:
import numpy as np

def find_optimal_k(X_train, y_train, X_val, y_val, max_k=15):
    """
    Evaluates CustomKNN across different odd values of K to find the 'elbow' (lowest error).
    """
    print(f"{'K Value':<10} | {'Validation Error':<20} | {'Accuracy':<10}")
    print("-" * 45)
    
    k_values = list(range(1, max_k + 2, 2)) # Tests 1, 3, 5, 7, 9, 11, 13, 15
    errors = []
    
    for k in k_values:
        # Initialize your custom KNN with the distance weights
        knn = CustomKNN(k=k, weights='distance')
        knn.fit(X_train, y_train)
        
        # Predict on validation set
        y_pred = knn.predict(X_val)
        
        # Calculate pure numpy error
        acc = np.sum(y_pred == y_val) / len(y_val)
        error = 1.0 - acc
        errors.append(error)
        
        print(f"{k:<10} | {error:<20.4f} | {acc:<10.4f}")
        
    best_idx = np.argmin(errors)
    best_k = k_values[best_idx]
    
    print("-" * 45)
    print(f"Optimal K found at K = {best_k} with Error = {errors[best_idx]:.4f}")
    
    return best_k

In [ ]:
y_knn_test = (y_test == 6).astype(int)

In [ ]:
print("=== KNN: Finding Optimal K ===")
best_k = find_optimal_k(X_train, y_train, X_val, y_val, max_k=15)

In [ ]:
subset_idx = np.random.RandomState(42).choice(len(X_train), 10000, replace=False)
X_sub, y_sub = X_train[subset_idx], y_train[subset_idx]

def train_knn(X, y):
    knn = CustomKNN(k=best_k, weights='distance')
    knn.fit(X, y)
    return knn

def predict_knn(model, X):
    return model.predict(X)

print(f"=== KNN K-Fold (k={best_k}, 5-fold on 10k subset) ===")
run_kfold(X_sub, y_sub, train_knn, predict_knn, k=5)

In [ ]:
print(f"=== Final KNN (k={best_k}, distance weighting) ===")
knn_final = CustomKNN(k=best_k, weights='distance')
knn_final.fit(X_train, y_train)

knn_preds = knn_final.predict(X_test_hog_pca)

print(classification_report(y_knn_test, knn_preds, target_names=["Not 6", "Is 6"]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_knn_test, knn_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Not 6", "Is 6"],
            yticklabels=["Not 6", "Is 6"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"KNN Confusion Matrix (k={best_k})")
plt.tight_layout()
plt.show()

In [ ]:
for w in ['uniform', 'distance']:
    knn_w = CustomKNN(k=best_k, weights=w)
    knn_w.fit(X_train, y_train)
    preds_w = knn_w.predict(X_test_hog_pca)
    f1 = f1_score(y_knn_test, preds_w, pos_label=1)
    acc = np.mean(preds_w == y_knn_test)
    print(f"Weighting={w:10s} → Accuracy={acc:.4f}, F1={f1:.4f}")